


---
#### 1. What is Delta Lake?
> "Delta Lake is a storage layer built on top of cloud object storage like ADLS. It provides features such as ACID transactions, schema enforcement, schema evolution, time travel and MERGE. In Databricks, I mainly use Delta tables for reliable batch and incremental data processing."

#### 2. Why Delta over Parquet?
> "Parquet is mainly a file format, whereas Delta adds a transaction layer on top of Parquet. Delta gives us ACID transactions, MERGE, schema enforcement, schema evolution, time travel and better support for incremental workloads. So for production data pipelines, Delta is more reliable than using plain Parquet."
>
> Good interview line:  
> "Parquet stores the data; Delta manages the data reliably."

#### 3. What is an ACID transaction in Delta?
> "ACID means Atomicity, Consistency, Isolation and Durability. In Delta, when we perform an operation like MERGE or UPDATE, the transaction either completes successfully or doesn't get committed. This prevents users from seeing partially written data."

#### 4. What is the Delta transaction log?
> "The Delta transaction log is the _delta_log directory associated with a Delta table. It keeps track of changes made to the table, such as file additions, removals, schema changes and transaction information. Delta uses this log to maintain consistency and provide features like time travel."
>
> You can visualize it as:
> 
> my_table/
> ├── part-0001.parquet
> ├── part-0002.parquet
> └── _delta_log/
>     ├── 000000.json
>     ├── 000001.json
>     └── 000002.json
> 

#### 5. Explain MERGE INTO.
> "MERGE is used when I need to perform both INSERT and UPDATE operations based on a matching condition. It's very useful for incremental loads and implementing SCD logic."
>
> Example:
> 
> MERGE INTO target t
> USING source s
> ON t.customer_id = s.customer_id
>  
> WHEN MATCHED THEN
>   UPDATE SET *
>  
> WHEN NOT MATCHED THEN
>   INSERT *
> 
> Then explain:  
> "If the customer already exists, it gets updated. If the customer doesn't exist, it gets inserted."

#### 6. How do you implement SCD Type 1 using Delta?
> SCD Type 1 means no history is maintained.
>
> Suppose:
>
> Target
> 
> customer_id | name   | city
> 101         | Shyam  | Pune
> 
> Source:
> 
> customer_id | name   | city
> 101         | Shyam  | Mumbai
> 
> We simply update Mumbai over Pune.
>
> 
> MERGE INTO customer_target t
> USING customer_source s
> ON t.customer_id = s.customer_id
>  
> WHEN MATCHED THEN
>   UPDATE SET
>     t.name = s.name,
>     t.city = s.city
>  
> WHEN NOT MATCHED THEN
>   INSERT (customer_id, name, city)
>   VALUES (s.customer_id, s.name, s.city);
> 
> Interview answer:  
> "For SCD Type 1, I use Delta MERGE. Existing records are updated and new records are inserted, so we don't maintain the previous value."

#### 7. How do you implement SCD Type 2?
> SCD Type 2 maintains history.
>
> Typical columns:
> - customer_id
> - name
> - city
> - effective_start_date
> - effective_end_date
> - is_current
>
> For example:
> 
> 101 | Shyam | Pune   | 2026-01-01 | 2026-08-18 | false
> 101 | Shyam | Mumbai | 2026-08-18 | null       | true
> 
> Interview answer:  
> "For SCD Type 2, when an existing record changes, I expire the old record by setting its end date and current flag to false, and then insert a new record with the updated values, new start date and is_current as true."
>
> If they ask "Can you implement this in Delta?", you should be prepared to write a more detailed MERGE or a staged update-and-insert approach.

#### 8. How do you perform incremental loads using Delta?
> "I first identify the new or changed records using a watermark column such as last_updated_timestamp, or use Delta Change Data Feed where applicable. Then I process only those records instead of reading the complete source every time. For the target, I use Delta MERGE to update existing records and insert new records."
>
> Example:
> 
> Source
>    ↓
> Identify changed records
>    ↓
> Bronze
>    ↓
> Silver MERGE
>    ↓
> Gold
> 

#### 9. What is schema enforcement?
> "Schema enforcement means Delta checks whether incoming data matches the existing table schema. If the incoming data has an incompatible schema, Delta can reject the write instead of silently corrupting the table."
>
> Example:
> Existing:
> 
> customer_id → INT
> 
> Incoming:
> 
> customer_id → STRING
> 
> Depending on the operation and compatibility, Delta can reject the write rather than automatically changing the table.

#### 10. What is schema evolution?
> "Schema evolution allows the schema of a Delta table to be changed to accommodate compatible changes in incoming data, such as adding a new column, when explicitly enabled/configured."
>
> Example:
> Existing:
> 
> customer_id
> name
> city
> 
> New data:
> 
> customer_id
> name
> city
> email
> 
> With appropriate schema evolution configuration, the email column can be added to the Delta table.
>
> Remember:  
> Schema enforcement protects the schema; schema evolution allows controlled schema changes.

#### 11. Difference between overwrite, append and merge
| Mode      | What happens                                   |
|-----------|------------------------------------------------|
| append    | Adds new records                               |
| overwrite | Replaces existing table/data according to the write operation |
| merge     | Updates matching records and inserts non-matching records      |

> Example:  
> "I use append when I'm only adding new data, overwrite when I intentionally want to replace the target data, and MERGE when I have both inserts and updates."

#### 12. What is Time Travel?
> "Delta Time Travel allows us to access previous versions of a Delta table. It's useful for auditing, debugging and recovering previous data states."
>
> Example:
> 
> SELECT *
> FROM customer
> VERSION AS OF 10;
> 
> Or using a timestamp:
> 
> SELECT *
> FROM customer
> TIMESTAMP AS OF '2026-08-17 10:00:00';
> 

#### 13. How do you restore an old version of a Delta table?
> "I can use the Delta RESTORE command to restore a table to a previous version or timestamp."
>
> Example:
> 
> RESTORE TABLE customer
> TO VERSION AS OF 10;
> 
> Important:  
> "The old version needs to remain available according to the table's retention settings. If the underlying files have already been permanently removed by VACUUM, that historical version may no longer be recoverable."

#### 14. What is OPTIMIZE?
> "OPTIMIZE is used to compact small files in a Delta table into larger files. This improves read performance by reducing the number of files Spark needs to scan."
>
> Example:
> 
> OPTIMIZE customer;
> 
> You can also optimize a subset using a predicate when appropriate.

#### 15. What is ZORDER?
> "ZORDER is a data layout optimization used with OPTIMIZE. It colocates related data based on specified columns, which can reduce the amount of data that needs to be read for selective queries."
>
> Example:
> 
> OPTIMIZE customer
> ZORDER BY (customer_id);
> 
> Good interview point:  
> "I would consider ZORDER for columns that are frequently used in filters, rather than blindly applying it to every column."

#### 16. What is VACUUM?
> "VACUUM removes old, unreferenced data files from a Delta table after the applicable retention period. It's mainly used for storage cleanup."
>
> Example:
> 
> VACUUM customer;
> 

#### 17. Difference between OPTIMIZE, ZORDER and VACUUM

| Feature   | Purpose                                           |
|-----------|---------------------------------------------------|
| OPTIMIZE  | Compacts small files                              |
| ZORDER    | Improves data layout for selective queries         |
| VACUUM    | Removes old unreferenced files                    |

> Simple way to remember:  
> OPTIMIZE = organize files  
> ZORDER   = organize data inside files for specific access patterns  
> VACUUM   = clean old files

#### 18. What happens if you run VACUUM?
> "VACUUM permanently deletes old files that are no longer referenced by the Delta table and are older than the configured retention threshold. After those files are deleted, corresponding historical versions may no longer be accessible through time travel."
>
> Very important:  
> "So I would be careful with VACUUM retention in production, especially if historical recovery or long-running readers are required."

#### 19. How do you handle small files?
> "First I identify the problem through table/file statistics or Spark performance. Then I look at the ingestion and write pattern because excessive small files are often created by frequent small writes. I can use OPTIMIZE to compact existing files and adjust the pipeline's partitioning or write strategy to prevent the problem from recurring."

#### 20. What is partitioning in Delta?
> "Partitioning physically organizes data into separate directory structures based on one or more columns. When a query filters on the partition column, Spark can potentially skip unrelated partitions, reducing the amount of data scanned."
>
> For example:
> 
> transactions/
>    year=2026/
>       month=08/
>    year=2025/
>       month=07/
> 

#### 21. When should you partition a Delta table?
> "I partition when the table is sufficiently large and queries frequently filter on a column with a reasonable number of distinct values, such as date. I avoid partitioning on high-cardinality columns like customer_id because it can create a large number of small files and partitions."
>
> Good answer:  
> "I don't partition every table by default. I choose partitioning based on data size, query patterns and cardinality."

#### 22. What is Change Data Feed?
> "Delta Change Data Feed, or CDF, allows us to track row-level changes made to a Delta table, such as inserts, updates and deletes. It is useful for incremental data processing because downstream pipelines can consume only the records that changed instead of scanning the complete table."
>
> Conceptually:
> 
> Delta Table
>      ↓
> Change Data Feed
>      ↓
> INSERT / UPDATE / DELETE
>      ↓
> Downstream processing
> 

#### 23. How would you process only changed records using CDF?
> First enable CDF on the table:
> 
> ALTER TABLE customer
> SET TBLPROPERTIES (
>   'delta.enableChangeDataFeed' = true
> );
> 
> Then read the changes:
> 
> changes_df = (
>     spark.read
>     .format("delta")
>     .option("readChangeFeed", "true")
>     .option("startingVersion", 10)
>     .table("catalog.schema.customer")
> )
> 
> Then process those changes downstream.
>
> You can inspect columns such as:
> - _change_type
> - _commit_version
> - _commit_timestamp
>
> For example:
> 
> customer_id | city    | _change_type
> 101         | Mumbai  | update_postimage
> 102         | Pune    | insert
> 103         | Delhi   | delete
> 
> Interview answer:  
> "I can use _change_type to understand whether the record was inserted, updated or deleted and then apply the appropriate operation to the downstream table."

---

### 🔥 Scenario: 10 million records every day, but only 2% are new/updated

> This is the answer I would particularly prepare.
>
> Interviewer:
> "You receive 10 million records every day, but only 2% are new or updated. How would you design the Delta pipeline?"
>
> Strong answer:
> "I wouldn't process all 10 million records through every layer unnecessarily. I would design an incremental pipeline. First, I would ingest the source into the Bronze Delta layer. Then I would identify the new or changed records using a reliable watermark such as last_updated_timestamp, or Delta Change Data Feed if the source is a Delta table. So instead of processing all 10 million records, I would process only the approximately 200,000 changed records. In Silver, I would apply cleansing, deduplication and business transformations and then use Delta MERGE based on the business key to update existing records and insert new records. Gold tables would then consume the required Silver changes. Finally, I would monitor file sizes and use OPTIMIZE where required."
>
> Architecture:
> 
> Source
>                  │
>                  ↓
>           10M records/day
>                  │
>                  ↓
>            Bronze Delta
>                  │
>                  ↓
>          Incremental detection
>         (watermark / CDF)
>                  │
>                  ↓
>         ~200K changed rows
>                  │
>                  ↓
>           Silver Delta
>        cleansing + validation
>        deduplication + MERGE
>                  │
>                  ↓
>            Gold Delta
>                  │
>                  ↓
>        Analytics / Reporting
> 
>
> If they ask: "Why not just process all 10 million?"
> > "Processing the full dataset every day would increase compute cost, processing time and unnecessary I/O. Since only 2% changes, incremental processing is much more efficient."
>
> If they ask: "How do you make sure you don't miss records?"
> > "I would use a reliable incremental mechanism such as a source watermark with proper checkpointing, or Delta CDF where applicable. I would also maintain the processing state and validate record counts between stages."
>
> If they ask: "What if the pipeline fails after processing 100,000 of those 200,000 records?"
> > "The pipeline should be designed to be restartable and idempotent. With checkpointing and appropriate MERGE logic, I can restart without creating duplicates or corrupting the target. I would also use workflow retries for transient failures."

---

### ⭐ The 10 Delta questions I would absolutely master

> For your experience level, make sure you can confidently explain these without hesitation:
> - Delta vs Parquet
> - Transaction log
> - MERGE
> - SCD Type 1
> - SCD Type 2
> - Incremental loading
> - Time Travel
> - OPTIMIZE / ZORDER / VACUUM
> - CDF
> - Designing a 10-million-record incremental pipeline
>
> And don't just memorize the definitions. The interviewer is very likely to follow up with "Show me the code", "Why did you choose that?", or "What happens if it fails?" after your first answer.

---